# Preparação dos Dados e Modelagem

## Projeto Integrador — Grupo 5
### Cidade Inteligente: Auditoria e Prevenção de Fraudes

**Responsável:** João Batista — FinOps / Machine Learning

## Objetivo

Preparar os dados e desenvolver experimentos de Machine Learning para identificação de transações potencialmente fraudulentas, utilizando o dataset sintético PaySim.

## Etapas do desenvolvimento

1. Carregamento e seleção dos dados.
2. Separação temporal entre treinamento, validação e teste.
3. Tratamento de outliers.
4. Binning e Feature Engineering.
5. Normalização e codificação.
6. Treinamento e avaliação do KNN.
7. Treinamento e avaliação do LightGBM.
8. Comparação dos modelos.
9. Avaliação final no conjunto de teste.

## Critérios metodológicos

- Preservar a separação temporal.
- Evitar Data Leakage.
- Considerar o desbalanceamento das classes.
- Ajustar transformações exclusivamente com os dados de treinamento.
- Manter o conjunto de teste reservado para avaliação final.

### ETAPA 1 — CARREGAMENTO DA BASE PARA MACHINE LEARNING

In [ ]:

# ============================================================
# ETAPA 1 — CARREGAMENTO DA BASE PARA MACHINE LEARNING
# ============================================================

import pandas as pd
import numpy as np

# Caminho do dataset original.
caminho_arquivo = "../data/raw/PS_20174392719_1491204439457_log.csv"

# Selecionamos apenas as colunas necessárias para nossos experimentos.
colunas_selecionadas = [
    'step',
    'type',
    'amount',
    'nameDest',
    'isFraud'
]

# Carregamos os dados com tipos definidos para reduzir
# o consumo de memória.
df_ml = pd.read_csv(
    caminho_arquivo,
    usecols=colunas_selecionadas,
    dtype={
        'step': 'int16',
        'type': 'category',
        'amount': 'float64',
        'nameDest': 'string',
        'isFraud': 'int8'
    }
)

# Criamos uma variável que identifica se o destinatário
# é um comerciante, conforme o prefixo utilizado no PaySim.
df_ml['destino_eh_comercio'] = (
    df_ml['nameDest']
    .str.startswith('M')
    .astype('int8')
)

# Removemos o identificador bruto do destinatário.
df_ml = df_ml.drop(columns=['nameDest'])

# Conferimos a estrutura da base.
print("Dimensões da base:", df_ml.shape)

print("\nColunas disponíveis:")
print(df_ml.columns.tolist())

print("\nTipos de dados:")
print(df_ml.dtypes)

display(df_ml.head())


### ETAPA 2 — SEPARAÇÃO TEMPORAL DOS DADOS

In [ ]:

# ============================================================
# ETAPA 2 — SEPARAÇÃO TEMPORAL DOS DADOS
# ============================================================

# Identifiquei o último step disponível no dataset.
step_maximo = int(df_ml['step'].max())

# Defini os limites temporais.
limite_treino = int(step_maximo * 0.70)
limite_validacao = int(step_maximo * 0.85)

# Classifiquei cada transação conforme seu período.
df_ml['conjunto'] = np.select(
    [
        df_ml['step'] <= limite_treino,
        df_ml['step'] <= limite_validacao
    ],
    [
        'treino',
        'validacao'
    ],
    default='teste'
)

# Resumi a quantidade de registros e fraudes por conjunto.
resumo_temporal = (
    df_ml.groupby('conjunto', observed=True)
    .agg(
        step_inicial=('step', 'min'),
        step_final=('step', 'max'),
        total_transacoes=('isFraud', 'size'),
        total_fraudes=('isFraud', 'sum')
    )
)

# Calculei a quantidade de transações legítimas.
resumo_temporal['total_legitimas'] = (
    resumo_temporal['total_transacoes']
    - resumo_temporal['total_fraudes']
)

# Calculei o percentual de fraudes.
resumo_temporal['taxa_fraude_pct'] = (
    resumo_temporal['total_fraudes']
    / resumo_temporal['total_transacoes']
    * 100
).round(4)

display(resumo_temporal)


### ETAPA 3 — SEPARAÇÃO DE FEATURES (X) E TARGET (y)

In [ ]:

# ============================================================
# ETAPA 3 — SEPARAÇÃO DE FEATURES (X) E TARGET (y)
# ============================================================

# Defini as variáveis que serão utilizadas como entrada dos modelos.
colunas_features = [
    'type',
    'amount',
    'destino_eh_comercio'
]

# Separei os registros por conjunto temporal.
mask_treino = df_ml['conjunto'] == 'treino'
mask_validacao = df_ml['conjunto'] == 'validacao'
mask_teste = df_ml['conjunto'] == 'teste'

# Criei os dados de entrada (X) para cada conjunto.
X_train = df_ml.loc[mask_treino, colunas_features].copy()
X_val = df_ml.loc[mask_validacao, colunas_features].copy()
X_test = df_ml.loc[mask_teste, colunas_features].copy()

# Criei as variáveis-alvo (y).
y_train = df_ml.loc[mask_treino, 'isFraud'].copy()
y_val = df_ml.loc[mask_validacao, 'isFraud'].copy()
y_test = df_ml.loc[mask_teste, 'isFraud'].copy()

# Conferi as dimensões de cada conjunto.
print("Treinamento:", X_train.shape, y_train.shape)
print("Validação:", X_val.shape, y_val.shape)
print("Teste:", X_test.shape, y_test.shape)

# Conferi a quantidade de fraudes e transações legítimas.
distribuicao_target = pd.DataFrame({
    'Treinamento': y_train.value_counts(),
    'Validação': y_val.value_counts(),
    'Teste': y_test.value_counts()
}).fillna(0).astype(int)

distribuicao_target.index = [
    'Legítimas' if classe == 0 else 'Fraudulentas'
    for classe in distribuicao_target.index
]

display(distribuicao_target)


### ETAPA 4 — DIAGNÓSTICO DE OUTLIERS NO TREINAMENTO

In [ ]:

# ============================================================
# ETAPA 4 — DIAGNÓSTICO DE OUTLIERS NO TREINAMENTO
# ============================================================

# Calculei os quartis da variável amount utilizando
# exclusivamente os dados de treinamento.

q1_amount = X_train['amount'].quantile(0.25)
q3_amount = X_train['amount'].quantile(0.75)

# Calculei a amplitude interquartil.
iqr_amount = q3_amount - q1_amount

# Defini os limites para identificação de outliers.
limite_inferior = q1_amount - (1.5 * iqr_amount)
limite_superior = q3_amount + (1.5 * iqr_amount)

# Identifiquei os valores que estão fora dos limites.
mascara_outliers = (
    (X_train['amount'] < limite_inferior) |
    (X_train['amount'] > limite_superior)
)

# Organizei os resultados por classe.
diagnostico_outliers = pd.DataFrame({
    'Classe': y_train.map({
        0: 'Legítimas',
        1: 'Fraudulentas'
    }),
    'Outlier': mascara_outliers.astype(int)
})

# Calculei a quantidade e o percentual de outliers por classe.
resumo_outliers = (
    diagnostico_outliers.groupby('Classe')
    .agg(
        Total_transacoes=('Outlier', 'size'),
        Total_outliers=('Outlier', 'sum')
    )
)

resumo_outliers['Percentual_outliers (%)'] = (
    resumo_outliers['Total_outliers']
    / resumo_outliers['Total_transacoes']
    * 100
).round(2)

# Exibi os limites calculados.
print(f"Q1: R$ {q1_amount:,.2f}")
print(f"Q3: R$ {q3_amount:,.2f}")
print(f"IQR: R$ {iqr_amount:,.2f}")
print(f"Limite inferior: R$ {limite_inferior:,.2f}")
print(f"Limite superior: R$ {limite_superior:,.2f}")

# Exibi a quantidade de outliers por classe.
display(resumo_outliers)


### ETAPA 5 — IMPACTO DA REMOÇÃO DE OUTLIERS

In [ ]:

# ============================================================
# ETAPA 5 — IMPACTO DA REMOÇÃO DE OUTLIERS
# ============================================================

# Identifiquei os registros que estão dentro dos limites do IQR.
mascara_sem_outliers = (
    X_train['amount'] <= limite_superior
)

# Calculei a quantidade de registros que permaneceria
# em cada classe caso os outliers fossem removidos.

impacto_remocao = pd.DataFrame({
    'Original': y_train.value_counts(),
    'Após remoção': y_train[mascara_sem_outliers].value_counts()
})

impacto_remocao = impacto_remocao.fillna(0).astype(int)

impacto_remocao.index = [
    'Legítimas' if classe == 0 else 'Fraudulentas'
    for classe in impacto_remocao.index
]

# Calculei quantos registros seriam perdidos.
impacto_remocao['Registros removidos'] = (
    impacto_remocao['Original']
    - impacto_remocao['Após remoção']
)

# Calculei o percentual de perda por classe.
impacto_remocao['Perda (%)'] = (
    impacto_remocao['Registros removidos']
    / impacto_remocao['Original']
    * 100
).round(2)

display(impacto_remocao)


### ETAPA 6 — TRATAMENTO DE OUTLIERS POR CAPPING

In [ ]:

# ============================================================
# ETAPA 6 — TRATAMENTO DE OUTLIERS POR CAPPING
# ============================================================

# Criei cópias independentes dos conjuntos originais.
X_train_cap = X_train.copy()
X_val_cap = X_val.copy()
X_test_cap = X_test.copy()

# Identifiquei quantos valores serão limitados em cada conjunto.
quantidade_cap = {
    'Treinamento': (X_train_cap['amount'] > limite_superior).sum(),
    'Validação': (X_val_cap['amount'] > limite_superior).sum(),
    'Teste': (X_test_cap['amount'] > limite_superior).sum()
}

# Apliquei o mesmo limite superior calculado no treinamento.
X_train_cap['amount'] = X_train_cap['amount'].clip(
    upper=limite_superior
)

X_val_cap['amount'] = X_val_cap['amount'].clip(
    upper=limite_superior
)

X_test_cap['amount'] = X_test_cap['amount'].clip(
    upper=limite_superior
)

# Conferi os resultados do tratamento.
resultado_capping = pd.DataFrame({
    'Conjunto': quantidade_cap.keys(),
    'Valores limitados': quantidade_cap.values(),
    'Total de registros': [
        len(X_train_cap),
        len(X_val_cap),
        len(X_test_cap)
    ],
    'Maior valor após tratamento': [
        X_train_cap['amount'].max(),
        X_val_cap['amount'].max(),
        X_test_cap['amount'].max()
    ]
})

display(resultado_capping)


### ETAPA 7 — BINNING DOS VALORES DAS TRANSAÇÕES

In [ ]:

# ============================================================
# ETAPA 7 — BINNING DOS VALORES DAS TRANSAÇÕES
# ============================================================

# Calculei os pontos de corte utilizando somente o treinamento.
# Os percentis 20%, 40%, 60% e 80% dividem a distribuição
# em até cinco faixas.

pontos_corte = (
    X_train_cap['amount']
    .quantile([0.20, 0.40, 0.60, 0.80])
    .to_numpy()
)

# Removi possíveis limites repetidos.
pontos_corte = np.unique(pontos_corte)

# Criei os limites das faixas.
# Valores abaixo ou acima dos limites também serão classificados.
limites_bins = np.concatenate((
    [-np.inf],
    pontos_corte,
    [np.inf]
))

# Criei os nomes das categorias.
rotulos_bins = [
    f'faixa_{i + 1}'
    for i in range(len(limites_bins) - 1)
]

# Apliquei os mesmos limites nos três conjuntos.
X_train_cap['faixa_valor'] = pd.cut(
    X_train_cap['amount'],
    bins=limites_bins,
    labels=rotulos_bins,
    include_lowest=True
)

X_val_cap['faixa_valor'] = pd.cut(
    X_val_cap['amount'],
    bins=limites_bins,
    labels=rotulos_bins,
    include_lowest=True
)

X_test_cap['faixa_valor'] = pd.cut(
    X_test_cap['amount'],
    bins=limites_bins,
    labels=rotulos_bins,
    include_lowest=True
)

# Exibi os limites encontrados.
print("Limites das faixas:")
for i in range(len(limites_bins) - 1):
    print(
        f"{rotulos_bins[i]}: "
        f"{limites_bins[i]:,.2f} até "
        f"{limites_bins[i + 1]:,.2f}"
    )

# Comparei a distribuição das faixas nos três conjuntos.
resumo_binning = pd.DataFrame({
    'Treinamento': X_train_cap['faixa_valor'].value_counts(
        sort=False
    ),
    'Validação': X_val_cap['faixa_valor'].value_counts(
        sort=False
    ),
    'Teste': X_test_cap['faixa_valor'].value_counts(
        sort=False
    )
})

display(resumo_binning)


### ETAPA 8 — FEATURE ENGINEERING: LOG DO VALOR

In [ ]:

# ============================================================
# ETAPA 8 — FEATURE ENGINEERING: LOG DO VALOR
# ============================================================

# Criei uma nova variável utilizando transformação logarítmica.
# A função log1p permite trabalhar também com valores iguais a zero.

X_train_cap['log_amount'] = np.log1p(
    X_train_cap['amount']
)

X_val_cap['log_amount'] = np.log1p(
    X_val_cap['amount']
)

X_test_cap['log_amount'] = np.log1p(
    X_test_cap['amount']
)

# Comparei as estatísticas antes e depois da transformação.

comparacao_log = pd.DataFrame({
    'Valor original': X_train_cap['amount'].describe(),
    'Valor transformado': X_train_cap['log_amount'].describe()
})

display(comparacao_log)


### ETAPA 9 — COMPARAÇÃO DA ASSIMETRIA (SKEWNESS)

In [ ]:

# ============================================================
# ETAPA 9 — COMPARAÇÃO DA ASSIMETRIA (SKEWNESS)
# ============================================================

# Calculei a assimetria dos valores originais
# e dos valores após a transformação logarítmica.

assimetria = pd.DataFrame({
    'Variável': ['amount', 'log_amount'],
    'Assimetria (Skewness)': [
        X_train_cap['amount'].skew(),
        X_train_cap['log_amount'].skew()
    ]
})

display(assimetria)


### ETAPA 10 — NORMALIZAÇÃO DAS VARIÁVEIS NUMÉRICAS

In [ ]:

# ============================================================
# ETAPA 10 — NORMALIZAÇÃO DAS VARIÁVEIS NUMÉRICAS
# ============================================================

from sklearn.preprocessing import StandardScaler

# Defini as variáveis numéricas do primeiro experimento.
colunas_numericas = [
    'log_amount'
]

# Inicializei o normalizador.
scaler = StandardScaler()

# Ajustei o normalizador somente com o treinamento.
X_train_knn = X_train_cap.copy()
X_val_knn = X_val_cap.copy()
X_test_knn = X_test_cap.copy()

X_train_knn[colunas_numericas] = scaler.fit_transform(
    X_train_knn[colunas_numericas]
)

# Apliquei a mesma transformação na validação e no teste.
X_val_knn[colunas_numericas] = scaler.transform(
    X_val_knn[colunas_numericas]
)

X_test_knn[colunas_numericas] = scaler.transform(
    X_test_knn[colunas_numericas]
)

# Conferi as estatísticas da variável normalizada.
display(
    X_train_knn['log_amount'].agg(
        ['mean', 'std', 'min', 'max']
    )
)


### ETAPA 11 — ONE-HOT ENCODING PARA O KNN

In [ ]:

# ============================================================
# ETAPA 11 — ONE-HOT ENCODING PARA O KNN
# ============================================================

from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack, csr_matrix

# Defini as variáveis categóricas.
colunas_categoricas = ['type', 'faixa_valor']

# Inicializei o codificador.
encoder_knn = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=True,
    dtype=np.float32
)

# Aprendi as categorias somente com o treinamento.
tipo_train = encoder_knn.fit_transform(
    X_train_cap[colunas_categoricas]
)

# Apliquei a mesma codificação na validação e no teste.
tipo_val = encoder_knn.transform(
    X_val_cap[colunas_categoricas]
)

tipo_test = encoder_knn.transform(
    X_test_cap[colunas_categoricas]
)

# Extraí as variáveis numéricas já normalizadas
# e a variável binária do destinatário.
def criar_features_numericas(base):
    return np.column_stack([
        base['log_amount'].to_numpy(dtype=np.float32),
        base['destino_eh_comercio'].to_numpy(dtype=np.float32)
    ])

numericas_train = criar_features_numericas(X_train_knn)
numericas_val = criar_features_numericas(X_val_knn)
numericas_test = criar_features_numericas(X_test_knn)

# Combinei as variáveis numéricas e categóricas
# em matrizes esparsas.
X_train_knn_final = hstack([
    csr_matrix(numericas_train),
    tipo_train
], format='csr')

X_val_knn_final = hstack([
    csr_matrix(numericas_val),
    tipo_val
], format='csr')

X_test_knn_final = hstack([
    csr_matrix(numericas_test),
    tipo_test
], format='csr')

# Conferi as dimensões finais.
print("Treinamento:", X_train_knn_final.shape)
print("Validação:", X_val_knn_final.shape)
print("Teste:", X_test_knn_final.shape)

print("\nQuantidade de features:", X_train_knn_final.shape[1])
print("Formato:", type(X_train_knn_final).__name__)


### ETAPA 12 — DIAGNÓSTICO DE MEMÓRIA DO KNN

In [ ]:

# ============================================================
# ETAPA 12 — DIAGNÓSTICO DE MEMÓRIA DO KNN
# ============================================================

# Avaliei o espaço ocupado pelas matrizes esparsas.

matrizes_knn = {
    'Treinamento': X_train_knn_final,
    'Validação': X_val_knn_final,
    'Teste': X_test_knn_final
}

for nome, matriz in matrizes_knn.items():

    # Somei a memória dos valores, índices e ponteiros.
    memoria_bytes = (
        matriz.data.nbytes
        + matriz.indices.nbytes
        + matriz.indptr.nbytes
    )

    # Converti para megabytes.
    memoria_mb = memoria_bytes / (1024 ** 2)

    print(f'{nome}:')
    print(f'Registros: {matriz.shape[0]:,}')
    print(f'Features: {matriz.shape[1]}')
    print(f'Memória: {memoria_mb:.2f} MB')
    print('-' * 40)


### ETAPA 13 — AMOSTRAGEM DO TREINAMENTO PARA KNN

In [ ]:

# ============================================================
# ETAPA 13 — AMOSTRAGEM DO TREINAMENTO PARA KNN
# ============================================================

# Configurei a proporção inicial de 10 legítimas para 1 fraude.
proporcao_legitimas = 10

# Identifiquei as posições de cada classe no treinamento.
indices_legitimas = np.flatnonzero(
    y_train.to_numpy() == 0
)

indices_fraudes = np.flatnonzero(
    y_train.to_numpy() == 1
)

# Calculei quantas transações legítimas serão selecionadas.
quantidade_legitimas = min(
    len(indices_legitimas),
    len(indices_fraudes) * proporcao_legitimas
)

# Realizei uma seleção aleatória e reproduzível.
gerador = np.random.default_rng(42)

indices_legitimas_amostra = gerador.choice(
    indices_legitimas,
    size=quantidade_legitimas,
    replace=False
)

# Preservei todas as fraudes.
# Combinei as posições das duas classes.
indices_knn = np.concatenate([
    indices_legitimas_amostra,
    indices_fraudes
])

# Embaralhei a ordem das observações.
gerador.shuffle(indices_knn)

# Criei a amostra final para treinamento do KNN.
X_train_knn_amostra = X_train_knn_final[indices_knn]
y_train_knn_amostra = (
    y_train.iloc[indices_knn].reset_index(drop=True)
)

# Conferi a distribuição das classes.
distribuicao_knn = pd.DataFrame({
    'Quantidade': y_train_knn_amostra.value_counts(),
    'Percentual (%)': (
        y_train_knn_amostra.value_counts(normalize=True) * 100
    ).round(2)
})

distribuicao_knn.index = [
    'Legítimas' if classe == 0 else 'Fraudulentas'
    for classe in distribuicao_knn.index
]

print("Dimensões da amostra KNN:", X_train_knn_amostra.shape)

display(distribuicao_knn)


### ETAPA 14 — TREINAMENTO DO MODELO KNN

In [ ]:

# ============================================================
# ETAPA 14 — TREINAMENTO DO MODELO KNN
# ============================================================

from sklearn.neighbors import KNeighborsClassifier

# Converti a matriz esparsa para matriz densa.
# Nossa matriz possui somente 12 features.

X_train_knn_dense = (
    X_train_knn_amostra.toarray()
    .astype(np.float32)
)

# Inicializei o modelo KNN.
knn_model = KNeighborsClassifier(
    n_neighbors=5,
    weights='distance',
    algorithm='kd_tree',
    leaf_size=40,
    metric='minkowski',
    p=2,
    n_jobs=-1
)

# Treinei o modelo com a amostra preparada.
knn_model.fit(
    X_train_knn_dense,
    y_train_knn_amostra.to_numpy()
)

print("Modelo KNN treinado com sucesso!")
print("Registros utilizados:", X_train_knn_dense.shape[0])
print("Quantidade de features:", X_train_knn_dense.shape[1])
print("Número de vizinhos:", knn_model.n_neighbors)
print("Método de ponderação:", knn_model.weights)


### ETAPA 15 — AVALIAÇÃO INICIAL DO KNN

In [ ]:

# ============================================================
# ETAPA 15 — AVALIAÇÃO INICIAL DO KNN
# ============================================================

import time

from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score
)

# Transformei a validação para o formato denso.
X_val_knn_dense = (
    X_val_knn_final.toarray()
    .astype(np.float32)
)

# Registrei o tempo inicial.
inicio = time.perf_counter()

# Calculei a probabilidade de cada transação ser fraude.
probabilidades_knn = knn_model.predict_proba(
    X_val_knn_dense
)[:, 1]

# Utilizei inicialmente o limiar de 0.5.
previsoes_knn = (
    probabilidades_knn >= 0.5
).astype(np.int8)

# Calculei as métricas de avaliação.
resultados_knn = {
    'Precision': precision_score(y_val, previsoes_knn),
    'Recall': recall_score(y_val, previsoes_knn),
    'F1-score': f1_score(y_val, previsoes_knn),
    'Average Precision (AP)': average_precision_score(
        y_val, probabilidades_knn
    ),
    'ROC-AUC': roc_auc_score(
        y_val, probabilidades_knn
    )
}

# Calculei o tempo de processamento.
tempo_execucao = time.perf_counter() - inicio

# Organizei as métricas em uma tabela.
metricas_knn = pd.DataFrame(
    resultados_knn.items(),
    columns=['Métrica', 'Resultado']
)

metricas_knn['Resultado'] = (
    metricas_knn['Resultado'].round(4)
)

print(f"Tempo de previsão e avaliação: {tempo_execucao:.2f} segundos")

display(metricas_knn)

# Analisei os acertos e erros do modelo.
matriz_confusao_knn = confusion_matrix(
    y_val,
    previsoes_knn,
    labels=[0, 1]
)

display(pd.DataFrame(
    matriz_confusao_knn,
    index=['Realmente legítima', 'Realmente fraudulenta'],
    columns=['Prevista legítima', 'Prevista fraudulenta']
))


### ETAPA 16 — COMPARAÇÃO DE LIMIARES DO KNN

In [ ]:

# ============================================================
# ETAPA 16 — COMPARAÇÃO DE LIMIARES DO KNN
# ============================================================

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Defini diferentes limiares para avaliar o KNN.
limiares = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50]

resultados_limiares = []

for limiar in limiares:

    # Classifiquei as transações utilizando o limiar atual.
    previsoes = (
        probabilidades_knn >= limiar
    ).astype(np.int8)

    # Calculei a matriz de confusão.
    tn, fp, fn, tp = confusion_matrix(
        y_val,
        previsoes,
        labels=[0, 1]
    ).ravel()

    # Armazenei as métricas de cada limiar.
    resultados_limiares.append({
        'Limiar': limiar,
        'Precision': precision_score(
            y_val, previsoes, zero_division=0
        ),
        'Recall': recall_score(
            y_val, previsoes, zero_division=0
        ),
        'F1-score': f1_score(
            y_val, previsoes, zero_division=0
        ),
        'Verdadeiros positivos': tp,
        'Falsos positivos': fp,
        'Falsos negativos': fn
    })

# Transformei os resultados em uma tabela.
comparacao_limiares = pd.DataFrame(resultados_limiares)

# Arredondei as métricas.
colunas_metricas = ['Precision', 'Recall', 'F1-score']

comparacao_limiares[colunas_metricas] = (
    comparacao_limiares[colunas_metricas].round(4)
)

display(comparacao_limiares)


### ETAPA 17 — CURVA PRECISION-RECALL DO KNN

In [ ]:

# ============================================================
# ETAPA 17 — CURVA PRECISION-RECALL DO KNN
# ============================================================

import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve

# Calculei Precision e Recall para diferentes limiares.
precisions, recalls, thresholds_pr = precision_recall_curve(
    y_val,
    probabilidades_knn
)

# Criei o gráfico.
plt.figure(figsize=(9, 6))

plt.plot(
    recalls,
    precisions,
    label='KNN - Precision-Recall'
)

# Identifiquei a proporção de fraudes na validação.
taxa_base = y_val.mean()

# Adicionei a linha de referência da classe positiva.
plt.axhline(
    y=taxa_base,
    linestyle='--',
    label='Referência da prevalência de fraudes'
)

plt.title('Curva Precision-Recall — KNN')
plt.xlabel('Recall')
plt.ylabel('Precision')

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


### Verificação das variáveis do experimento KNN

In [ ]:
# ============================================================
# VERIFICAÇÃO DAS VARIÁVEIS DO EXPERIMENTO KNN
# ============================================================

print("df_ml carregado:", 'df_ml' in globals())
print("y_val disponível:", 'y_val' in globals())
print("Modelo KNN disponível:", 'knn_model' in globals())
print("Probabilidades disponíveis:", 'probabilidades_knn' in globals())

### ETAPA 18 — VERIFICAÇÃO DA BIBLIOTECA LIGHTGBM

In [ ]:

# ============================================================
# ETAPA 18 — VERIFICAÇÃO DA BIBLIOTECA LIGHTGBM
# ============================================================

import importlib.util

# Verifiquei se a biblioteca LightGBM está instalada.
biblioteca_lightgbm = importlib.util.find_spec("lightgbm")

if biblioteca_lightgbm is not None:
    import lightgbm as lgb

    print("LightGBM instalado com sucesso!")
    print("Versão:", lgb.__version__)

else:
    print("LightGBM ainda não está instalado.")


### ETAPA 19 — PREPARAÇÃO DAS FEATURES PARA LIGHTGBM

In [ ]:

# ============================================================
# ETAPA 19 — PREPARAÇÃO DAS FEATURES PARA LIGHTGBM
# ============================================================

import lightgbm as lgb

# Defini as variáveis que serão utilizadas pelo LightGBM.
colunas_lgb = [
    'amount',
    'type',
    'destino_eh_comercio',
    'faixa_valor'
]

# Criei cópias dos conjuntos com Capping e Binning.
X_train_lgb = X_train_cap[colunas_lgb].copy()
X_val_lgb = X_val_cap[colunas_lgb].copy()
X_test_lgb = X_test_cap[colunas_lgb].copy()

# Identifiquei as variáveis categóricas.
colunas_categoricas_lgb = [
    'type',
    'faixa_valor'
]

# Garanti que as categorias da validação e do teste
# sejam exatamente as aprendidas no treinamento.
for coluna in colunas_categoricas_lgb:

    X_train_lgb[coluna] = (
        X_train_lgb[coluna].astype('category')
    )

    categorias_treino = (
        X_train_lgb[coluna].cat.categories
    )

    X_val_lgb[coluna] = pd.Categorical(
        X_val_lgb[coluna],
        categories=categorias_treino
    )

    X_test_lgb[coluna] = pd.Categorical(
        X_test_lgb[coluna],
        categories=categorias_treino
    )

# Conferi as dimensões dos conjuntos.
print("Treinamento:", X_train_lgb.shape)
print("Validação:", X_val_lgb.shape)
print("Teste:", X_test_lgb.shape)

# Conferi os tipos de dados.
print("\nTipos das variáveis:")
display(X_train_lgb.dtypes)

# Verifiquei a existência de valores ausentes.
print("\nValores ausentes no treinamento:")
display(X_train_lgb.isnull().sum())


### ETAPA 20 — CÁLCULO DO PESO DA CLASSE FRAUDULENTA

In [ ]:

# ============================================================
# ETAPA 20 — CÁLCULO DO PESO DA CLASSE FRAUDULENTA
# ============================================================

# Calculei a quantidade de registros de cada classe.
quantidade_legitimas = int((y_train == 0).sum())
quantidade_fraudes = int((y_train == 1).sum())

# Calculei a proporção entre as classes.
scale_pos_weight = (
    quantidade_legitimas / quantidade_fraudes
)

# Calculei também a raiz quadrada da proporção,
# que poderá ser utilizada em um experimento alternativo.
peso_alternativo = np.sqrt(scale_pos_weight)

# Organizei os resultados.
analise_pesos = pd.DataFrame({
    'Indicador': [
        'Transações legítimas',
        'Transações fraudulentas',
        'Proporção legítimas/fraudes',
        'Peso proporcional completo',
        'Peso alternativo (raiz quadrada)'
    ],
    'Resultado': [
        quantidade_legitimas,
        quantidade_fraudes,
        round(scale_pos_weight, 2),
        round(scale_pos_weight, 2),
        round(peso_alternativo, 2)
    ]
})

display(analise_pesos)


### ETAPA 21 — TREINAMENTO DO PRIMEIRO LIGHTGBM

In [ ]:

# ============================================================
# ETAPA 21 — TREINAMENTO DO PRIMEIRO LIGHTGBM
# ============================================================

import time
import lightgbm as lgb

# Inicializei o cronômetro.
inicio_treinamento = time.perf_counter()

# Configurei o modelo.
lgb_model = lgb.LGBMClassifier(
    objective='binary',
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=100,
    scale_pos_weight=scale_pos_weight,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

# Treinei utilizando todos os registros do treinamento.
# A validação será utilizada para acompanhar o desempenho
# e interromper o treinamento quando não houver melhoria.

lgb_model.fit(
    X_train_lgb,
    y_train,
    eval_set=[(X_val_lgb, y_val)],
    eval_metric='average_precision',
    categorical_feature='auto',
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

# Calculei o tempo de treinamento.
tempo_treinamento = (
    time.perf_counter() - inicio_treinamento
)

# Exibi as informações do modelo.
print("\nTreinamento concluído!")

print("Registros utilizados:", len(X_train_lgb))
print("Features utilizadas:", X_train_lgb.shape[1])
print("Peso da classe fraudulenta:", round(scale_pos_weight, 2))
print("Melhor iteração:", lgb_model.best_iteration_)
print("Melhor resultado:", lgb_model.best_score_)
print(f"Tempo de treinamento: {tempo_treinamento:.2f} segundos")


### ETAPA 22 — AVALIAÇÃO INICIAL DO LIGHTGBM

In [ ]:

# ============================================================
# ETAPA 22 — AVALIAÇÃO INICIAL DO LIGHTGBM
# ============================================================

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
    confusion_matrix
)

import pandas as pd
import numpy as np

# Calculei as probabilidades de fraude na validação.
probabilidades_lgb = lgb_model.predict_proba(
    X_val_lgb
)[:, 1]

# Utilizei inicialmente o limiar de 0.50.
previsoes_lgb = (
    probabilidades_lgb >= 0.50
).astype(np.int8)

# Calculei as métricas.
resultados_lgb = {
    'Precision': precision_score(
        y_val, previsoes_lgb, zero_division=0
    ),
    'Recall': recall_score(
        y_val, previsoes_lgb, zero_division=0
    ),
    'F1-score': f1_score(
        y_val, previsoes_lgb, zero_division=0
    ),
    'Average Precision': average_precision_score(
        y_val, probabilidades_lgb
    ),
    'ROC-AUC': roc_auc_score(
        y_val, probabilidades_lgb
    )
}

# Organizei os resultados.
metricas_lgb = pd.DataFrame(
    resultados_lgb.items(),
    columns=['Métrica', 'Resultado']
)

metricas_lgb['Resultado'] = (
    metricas_lgb['Resultado'].round(4)
)

display(metricas_lgb)

# Construí a matriz de confusão.
matriz_lgb = confusion_matrix(
    y_val,
    previsoes_lgb,
    labels=[0, 1]
)

display(pd.DataFrame(
    matriz_lgb,
    index=['Realmente legítima', 'Realmente fraudulenta'],
    columns=['Prevista legítima', 'Prevista fraudulenta']
))


### ETAPA 23 — DIAGNÓSTICO DAS PROBABILIDADES DO LIGHTGBM

In [ ]:

# ============================================================
# ETAPA 23 — DIAGNÓSTICO DAS PROBABILIDADES DO LIGHTGBM
# ============================================================

from sklearn.metrics import average_precision_score

# Gereis novamente as probabilidades usando explicitamente
# a melhor iteração encontrada durante o treinamento.
probabilidades_lgb_best = lgb_model.predict_proba(
    X_val_lgb,
    num_iteration=lgb_model.best_iteration_
)[:, 1]

# Comparei o AP registrado pelo LightGBM com o AP do Scikit-learn.
ap_sklearn_best = average_precision_score(
    y_val,
    probabilidades_lgb_best
)

ap_nativo = lgb_model.best_score_['valid_0']['average_precision']

# Resumi as probabilidades por classe real.
diagnostico_probabilidades = pd.DataFrame({
    'Probabilidade': probabilidades_lgb_best,
    'Classe real': y_val.to_numpy()
})

resumo_probabilidades = (
    diagnostico_probabilidades
    .groupby('Classe real')['Probabilidade']
    .agg(['min', 'mean', 'median', 'max'])
    .round(4)
)

# Verifiquei quantas transações seriam sinalizadas
# em diferentes limiares.
limiares_diagnostico = [0.01, 0.05, 0.10, 0.20, 0.50, 0.90]

resumo_limiares_lgb = pd.DataFrame({
    'Limiar': limiares_diagnostico,
    'Transações sinalizadas': [
        int((probabilidades_lgb_best >= limiar).sum())
        for limiar in limiares_diagnostico
    ]
})

print("Melhor iteração:", lgb_model.best_iteration_)
print("AP registrado pelo LightGBM:", round(ap_nativo, 6))
print("AP recalculado pelo Scikit-learn:", round(ap_sklearn_best, 6))

print("\nProbabilidades por classe real:")
display(resumo_probabilidades)

print("\nQuantidade de alertas por limiar:")
display(resumo_limiares_lgb)


### ETAPA 24 — QUANTIDADE DE ALERTAS POR LIMIAR

In [ ]:
# ============================================================
# ETAPA 24 — QUANTIDADE DE ALERTAS POR LIMIAR
# ============================================================

display(resumo_limiares_lgb)

### ETAPA 25 — COMPARAÇÃO DOS LIMIARES DO LIGHTGBM

In [ ]:

# ============================================================
# ETAPA 25 — COMPARAÇÃO DOS LIMIARES DO LIGHTGBM
# ============================================================

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Defini os limiares para comparação.
limiares_lgb = [0.50, 0.70, 0.80, 0.90, 0.95, 0.99]

resultados_limiares_lgb = []

for limiar in limiares_lgb:

    # Classifiquei as transações usando o limiar atual.
    previsoes = (
        probabilidades_lgb_best >= limiar
    ).astype(np.int8)

    # Calculei os componentes da matriz de confusão.
    tn, fp, fn, tp = confusion_matrix(
        y_val,
        previsoes,
        labels=[0, 1]
    ).ravel()

    # Calculei as métricas.
    resultados_limiares_lgb.append({
        'Limiar': limiar,
        'Precision': precision_score(
            y_val, previsoes, zero_division=0
        ),
        'Recall': recall_score(
            y_val, previsoes, zero_division=0
        ),
        'F1-score': f1_score(
            y_val, previsoes, zero_division=0
        ),
        'Verdadeiros positivos': tp,
        'Falsos positivos': fp,
        'Falsos negativos': fn,
        'Total de alertas': tp + fp
    })

# Organizei os resultados.
comparacao_limiares_lgb = pd.DataFrame(
    resultados_limiares_lgb
)

# Arredondei as métricas.
colunas_metricas = [
    'Precision',
    'Recall',
    'F1-score'
]

comparacao_limiares_lgb[colunas_metricas] = (
    comparacao_limiares_lgb[colunas_metricas].round(4)
)

display(comparacao_limiares_lgb)


### ETAPA 26 — DIAGNÓSTICO INTERNO DO LIGHTGBM

In [ ]:

# ============================================================
# ETAPA 26 — DIAGNÓSTICO INTERNO DO LIGHTGBM
# ============================================================

# 1. Analisei a distribuição das probabilidades por classe real.

valores_reais = y_val.to_numpy()

resumo_probabilidades_lgb = []

for classe in [0, 1]:

    probabilidades_classe = probabilidades_lgb_best[
        valores_reais == classe
    ]

    resumo_probabilidades_lgb.append({
        'Classe': 'Legítimas' if classe == 0 else 'Fraudulentas',
        'Quantidade': len(probabilidades_classe),
        'Mínimo': np.min(probabilidades_classe),
        'Mediana': np.median(probabilidades_classe),
        'Percentil 90': np.percentile(probabilidades_classe, 90),
        'Percentil 95': np.percentile(probabilidades_classe, 95),
        'Percentil 99': np.percentile(probabilidades_classe, 99),
        'Máximo': np.max(probabilidades_classe),
        'Acima de 0.99 (%)': (
            np.mean(probabilidades_classe >= 0.99) * 100
        )
    })

display(pd.DataFrame(resumo_probabilidades_lgb).round(4))


# 2. Verifiquei quantas árvores foram utilizadas.

print("\nINFORMAÇÕES DO MODELO")
print("Melhor iteração:", lgb_model.best_iteration_)
print("Total de árvores:", lgb_model.booster_.num_trees())
print("Total de features:", len(lgb_model.feature_name_))


# 3. Analisei a importância das variáveis.

importancias_lgb = pd.DataFrame({
    'Variável': lgb_model.feature_name_,
    'Importância (Gain)': lgb_model.booster_.feature_importance(
        importance_type='gain'
    ),
    'Importância (Split)': lgb_model.booster_.feature_importance(
        importance_type='split'
    )
})

importancias_lgb = importancias_lgb.sort_values(
    by='Importância (Gain)',
    ascending=False
)

display(importancias_lgb)


### ETAPA 27 — COMPARAÇÃO DOS PESOS DO LIGHTGBM

In [ ]:

# ============================================================
# ETAPA 27 — COMPARAÇÃO DOS PESOS DO LIGHTGBM
# ============================================================

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score
)

import time

# Guardei o modelo com ponderação integral, já treinado.
experimentos_lgb = {
    'Peso integral': {
        'modelo': lgb_model,
        'probabilidades': probabilidades_lgb_best,
        'peso': scale_pos_weight
    }
}

# Defini os dois novos experimentos.
pesos_experimentais = {
    'Sem ponderação': 1.0,
    'Peso intermediário': peso_alternativo
}

# Treinei os dois modelos adicionais.
for nome, peso in pesos_experimentais.items():

    print(f"\nTreinando: {nome} | Peso: {peso:.2f}")

    modelo = lgb.LGBMClassifier(
        objective='binary',
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=100,
        scale_pos_weight=peso,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    inicio = time.perf_counter()

    modelo.fit(
        X_train_lgb,
        y_train,
        eval_set=[(X_val_lgb, y_val)],
        eval_metric='average_precision',
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=50,
                first_metric_only=True,
                verbose=False
            )
        ]
    )

    tempo = time.perf_counter() - inicio

    probabilidades = modelo.predict_proba(
        X_val_lgb,
        num_iteration=modelo.best_iteration_
    )[:, 1]

    experimentos_lgb[nome] = {
        'modelo': modelo,
        'probabilidades': probabilidades,
        'peso': peso,
        'tempo': tempo
    }

    print("Melhor iteração:", modelo.best_iteration_)
    print(f"Tempo: {tempo:.2f} segundos")


# Comparei os três experimentos usando a mesma validação.
resultados_pesos = []

for nome, experimento in experimentos_lgb.items():

    probabilidades = experimento['probabilidades']

    previsoes = (
        probabilidades >= 0.50
    ).astype(np.int8)

    resultados_pesos.append({
        'Experimento': nome,
        'Peso': round(experimento['peso'], 2),
        'Precision': precision_score(
            y_val, previsoes, zero_division=0
        ),
        'Recall': recall_score(
            y_val, previsoes, zero_division=0
        ),
        'F1-score': f1_score(
            y_val, previsoes, zero_division=0
        ),
        'Average Precision': average_precision_score(
            y_val, probabilidades
        ),
        'ROC-AUC': roc_auc_score(
            y_val, probabilidades
        ),
        'Iteração': experimento['modelo'].best_iteration_
    })

comparacao_pesos_lgb = pd.DataFrame(resultados_pesos)

display(comparacao_pesos_lgb.round(4))


### ETAPA 28 — COMPARAÇÃO DE LIMIARES DO LIGHTGBM

In [ ]:

# ============================================================
# ETAPA 28 — COMPARAÇÃO DE LIMIARES DO LIGHTGBM
# ============================================================

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Separei as duas configurações mais promissoras.
modelos_comparacao = {
    'Sem ponderação': experimentos_lgb['Sem ponderação'],
    'Peso intermediário': experimentos_lgb['Peso intermediário']
}

# Defini os limiares que serão avaliados.
limiares = [
    0.01, 0.05, 0.10, 0.20, 0.30,
    0.40, 0.50, 0.60, 0.70, 0.80
]

resultados_comparacao = []

# Avaliei cada modelo com os mesmos limiares.
for nome_modelo, experimento in modelos_comparacao.items():

    probabilidades = experimento['probabilidades']

    for limiar in limiares:

        # Classifiquei as transações.
        previsoes = (
            probabilidades >= limiar
        ).astype(np.int8)

        # Calculei a matriz de confusão.
        tn, fp, fn, tp = confusion_matrix(
            y_val,
            previsoes,
            labels=[0, 1]
        ).ravel()

        # Calculei as métricas.
        resultados_comparacao.append({
            'Modelo': nome_modelo,
            'Limiar': limiar,
            'Precision': precision_score(
                y_val, previsoes, zero_division=0
            ),
            'Recall': recall_score(
                y_val, previsoes, zero_division=0
            ),
            'F1-score': f1_score(
                y_val, previsoes, zero_division=0
            ),
            'Verdadeiros positivos': tp,
            'Falsos positivos': fp,
            'Falsos negativos': fn,
            'Total de alertas': tp + fp
        })

# Organizei os resultados.
comparacao_limiares_final = pd.DataFrame(
    resultados_comparacao
)

# Arredondei as métricas.
colunas_metricas = [
    'Precision',
    'Recall',
    'F1-score'
]

comparacao_limiares_final[colunas_metricas] = (
    comparacao_limiares_final[colunas_metricas].round(4)
)

# Exibi a comparação.
display(comparacao_limiares_final)


### ETAPA 29 — OTIMIZAÇÃO DO LIMIAR DO LIGHTGBM

In [ ]:

# ============================================================
# ETAPA 29 — OTIMIZAÇÃO DO LIMIAR DO LIGHTGBM
# ============================================================

from sklearn.metrics import (
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Separei as duas configurações candidatas.
modelos_candidatos = {
    'Sem ponderação': (
        experimentos_lgb['Sem ponderação']['probabilidades']
    ),
    'Peso intermediário': (
        experimentos_lgb['Peso intermediário']['probabilidades']
    )
}

resultados_otimizacao = []

for nome, probabilidades in modelos_candidatos.items():

    # Calculei Precision, Recall e os possíveis limiares.
    precisions, recalls, thresholds = precision_recall_curve(
        y_val,
        probabilidades
    )

    # Calculei o F1-score para cada limiar.
    f1_scores = (
        2 * precisions[:-1] * recalls[:-1]
        / (precisions[:-1] + recalls[:-1] + 1e-12)
    )

    # Identifiquei o limiar que apresentou o maior F1-score.
    melhor_indice = np.argmax(f1_scores)
    melhor_limiar = thresholds[melhor_indice]

    # Gereis as previsões utilizando o limiar selecionado.
    previsoes = (
        probabilidades >= melhor_limiar
    ).astype(np.int8)

    # Calculei a matriz de confusão.
    tn, fp, fn, tp = confusion_matrix(
        y_val,
        previsoes,
        labels=[0, 1]
    ).ravel()

    # Armazenei os resultados.
    resultados_otimizacao.append({
        'Modelo': nome,
        'Melhor limiar': melhor_limiar,
        'Precision': precision_score(
            y_val, previsoes, zero_division=0
        ),
        'Recall': recall_score(
            y_val, previsoes, zero_division=0
        ),
        'F1-score': f1_score(
            y_val, previsoes, zero_division=0
        ),
        'Verdadeiros positivos': tp,
        'Falsos positivos': fp,
        'Falsos negativos': fn,
        'Total de alertas': tp + fp
    })

# Organizei a comparação.
otimizacao_limiares_lgb = pd.DataFrame(
    resultados_otimizacao
)

# Formatei as métricas.
display(otimizacao_limiares_lgb.round(4))


### ETAPA 30 — OTIMIZAÇÃO DO LIMIAR DO KNN

In [ ]:

# ============================================================
# ETAPA 30 — OTIMIZAÇÃO DO LIMIAR DO KNN
# ============================================================

from sklearn.metrics import (
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Calculei a curva Precision-Recall do KNN.
precisions_knn, recalls_knn, thresholds_knn = (
    precision_recall_curve(
        y_val,
        probabilidades_knn
    )
)

# Calculei o F1-score para cada limiar disponível.
f1_knn = (
    2 * precisions_knn[:-1] * recalls_knn[:-1]
    / (
        precisions_knn[:-1]
        + recalls_knn[:-1]
        + 1e-12
    )
)

# Identifiquei o melhor F1-score.
melhor_indice_knn = np.argmax(f1_knn)

melhor_limiar_knn = thresholds_knn[melhor_indice_knn]

# Gereis as previsões com o limiar selecionado.
previsoes_knn_otimizado = (
    probabilidades_knn >= melhor_limiar_knn
).astype(np.int8)

# Calculei a matriz de confusão.
tn, fp, fn, tp = confusion_matrix(
    y_val,
    previsoes_knn_otimizado,
    labels=[0, 1]
).ravel()

# Organizei os resultados.
resultado_knn_otimizado = pd.DataFrame({
    'Métrica': [
        'Melhor limiar',
        'Precision',
        'Recall',
        'F1-score',
        'Verdadeiros positivos',
        'Falsos positivos',
        'Falsos negativos',
        'Total de alertas'
    ],
    'Resultado': [
        melhor_limiar_knn,
        precision_score(y_val, previsoes_knn_otimizado),
        recall_score(y_val, previsoes_knn_otimizado),
        f1_score(y_val, previsoes_knn_otimizado),
        tp,
        fp,
        fn,
        tp + fp
    ]
})

display(resultado_knn_otimizado.round(4))


### ETAPA 31 — COMPARAÇÃO FINAL DOS MODELOS NA VALIDAÇÃO

In [ ]:

# ============================================================
# ETAPA 31 — COMPARAÇÃO FINAL DOS MODELOS NA VALIDAÇÃO
# ============================================================

# Organizei os resultados dos dois modelos
# utilizando os limiares otimizados na validação.

comparacao_modelos = pd.DataFrame([
    {
        'Modelo': 'KNN',
        'Limiar': melhor_limiar_knn,
        'Precision': precision_score(
            y_val, previsoes_knn_otimizado
        ),
        'Recall': recall_score(
            y_val, previsoes_knn_otimizado
        ),
        'F1-score': f1_score(
            y_val, previsoes_knn_otimizado
        ),
        'AP': average_precision_score(
            y_val, probabilidades_knn
        ),
        'ROC-AUC': roc_auc_score(
            y_val, probabilidades_knn
        )
    },
    {
        'Modelo': 'LightGBM sem ponderação',
        'Limiar': otimizacao_limiares_lgb.loc[
            otimizacao_limiares_lgb['Modelo'] == 'Sem ponderação',
            'Melhor limiar'
        ].iloc[0],
        'Precision': precision_score(
            y_val,
            (experimentos_lgb['Sem ponderação']['probabilidades']
             >= otimizacao_limiares_lgb.loc[
                 otimizacao_limiares_lgb['Modelo'] == 'Sem ponderação',
                 'Melhor limiar'
             ].iloc[0]).astype(np.int8)
        ),
        'Recall': recall_score(
            y_val,
            (experimentos_lgb['Sem ponderação']['probabilidades']
             >= otimizacao_limiares_lgb.loc[
                 otimizacao_limiares_lgb['Modelo'] == 'Sem ponderação',
                 'Melhor limiar'
             ].iloc[0]).astype(np.int8)
        ),
        'F1-score': f1_score(
            y_val,
            (experimentos_lgb['Sem ponderação']['probabilidades']
             >= otimizacao_limiares_lgb.loc[
                 otimizacao_limiares_lgb['Modelo'] == 'Sem ponderação',
                 'Melhor limiar'
             ].iloc[0]).astype(np.int8)
        ),
        'AP': average_precision_score(
            y_val,
            experimentos_lgb['Sem ponderação']['probabilidades']
        ),
        'ROC-AUC': roc_auc_score(
            y_val,
            experimentos_lgb['Sem ponderação']['probabilidades']
        )
    }
])

# Arredondei os resultados para facilitar a apresentação.
display(comparacao_modelos.round(4))
